In [1]:
from urllib.request import urlretrieve
from pathlib import Path
from zipfile import ZipFile
import shutil
import pandas as pd

In [2]:
ROOT = Path("~/code/selector_examenes").expanduser()
Path.mkdir(Path("examenes_quimica"), exist_ok=True)
EXAMS_PATH = Path("../examenes_quimica")

PARENT_URL = "https://www.juntadeandalucia.es/economiaconocimientoempresasyuniversidad/sguit/examanes_anios_anteriores/selectividad/"
# sel_2010_quimica.zip

## Descargar y extraer archivos de exámenes

### Descargar archivos

In [3]:
# for year in range(2000, 2026):

#     try:
#         urlretrieve(f"{PARENT_URL}sel_{year}_quimica.zip", ROOT / f"quimica_zip_files/{year}_quimica.zip")
#     except:
#         print(f"Año sin archivo: {year}")

### Extraer archivos PDF

Borrar versión de `examenes_quimica` anterior y crearla de nuevo

In [4]:
if Path(ROOT / "examenes_quimica").exists():
    shutil.rmtree(ROOT / "examenes_quimica")
    
Path.mkdir(ROOT / "examenes_quimica", exist_ok=True)

In [5]:
def extract_files_from_zip(zipfile: Path) -> None:
    extract_path = Path(ROOT / f"examenes_quimica/{zipfile.stem}")
    with ZipFile(zipfile, "r", metadata_encoding="cp850") as file:
        file.extractall(extract_path)
    criteria_files = [i for i in extract_path.rglob("*.pdf") if "criterios" in i.name.lower()]
    for f in criteria_files:
        f.unlink(missing_ok=True)

In [6]:
for year in range(2010, 2027):
    extract_files_from_zip(ROOT / f"quimica_zip_files/{year}_quimica.zip")

## Limpiar estructura de archivos

In [7]:
def get_files(path = EXAMS_PATH):
    return sorted(path.rglob("*.pdf"))

### Extraer archivos de subcarpetas

In [8]:
for folder in EXAMS_PATH.iterdir():
    if folder.is_dir():
        # find pdf files
        pdf_files = [f for f in folder.rglob("*.pdf")]

        for f in pdf_files:

            # get subfolders of pdf files
            subfolder = str(f.relative_to(folder).parent).replace("/", " - ")

            # move them into folder root
            dest = EXAMS_PATH / folder/ f"{subfolder} - {f.name}"

            # move the file to folder base
            shutil.move(f, dest)
            
        # remove subfolders
        for dir in folder.iterdir():
                if dir.is_dir():
                    shutil.rmtree(dir)

### Borrar archivos de criterios

In [9]:
for file in get_files():
    new_file = file.name.lower()

    # remove file if it is a criteria file
    if "criterio" in file.name.lower() or "_c_" in file.name.lower():
        file.unlink(missing_ok=True)
        continue

### Añadir año y asignatura al nombre de archivo

In [10]:
for file in get_files():
    filename_parts = list(file.parts)
    filename_parts[-1] = file.parts[-2] + "_" + filename_parts[-1]
    new_filename = "/".join(filename_parts)
    
    shutil.move(file,new_filename)

## Crear lista de nombres de archivo

In [11]:
chemistry_source_files =sorted(EXAMS_PATH.rglob("*.pdf"))

In [12]:
with open("chemistry_exam_files.txt", "w", encoding="utf-8") as f:
    for file in chemistry_source_files:
        f.write(f"{str(file)}\n")

In [13]:
pd.DataFrame({"source_pdf_files" : chemistry_source_files}).to_csv("chemistry_exam_files.csv", index=False, encoding="utf-8")

## Excepción para 2008
extraer y añadir año y asignatura al nombre de archivo

In [14]:
# zipfile = ROOT / "quimica_zip_files/2008_quimica.zip"
# extract_path = ROOT / f"examenes_quimica/2008_quimica"
# Path.mkdir(extract_path, exist_ok=True)

# with ZipFile(zipfile, "r") as file:
#     file.extractall(extract_path)

In [15]:
# for file in extract_path.iterdir():
#     filename_parts = list(file.parts)
#     filename_parts[-1] = file.parts[-2] + "_" + filename_parts[-1]
#     new_filename = "/".join(filename_parts)
    
#     shutil.move(file,new_filename)

## Limpiar nombres de archivo

Convertir a minúsculas, eliminar espacios extra y convertir guiones y espacios a guión bajo.

In [16]:
# for file in get_files():
#     filename = file.name
#     # eliminar espacios extra
#     filename = re.sub(r"\s{2,}", " ", filename)
#     filename = (filename
#                 .replace(" ", "_")
#                 .replace("-", "_")
#                 .lower()
#     )

#     # tras la sustitución, eliminar guiones bajos duplicados
#     filename = re.sub(r"_{2,}", "_", filename)

#     # corregir variantes de química
#     variants = ["química", "quimica", "qu╓mica", "quínica"]
#     pattern = "|".join(map(re.escape, variants))
#     filename = re.sub(pattern, "química", filename)

#     # eliminar la palabra química
#     filename = filename.replace("química", "")

#     # reemplazar estas strings por ""
#     strings_to_clean = ['exámenes', "cabe", "andalucía", "anda", "examen", "impreso", "corregido", "rev", "20_21"]
#     pattern = "|".join(map(re.escape, strings_to_clean))
#     filename = re.sub(pattern, "", filename)

#     # eliminar números al final del nombre
#     filename = re.sub(r"_\d+.pdf$", ".pdf", filename)

#     # eliminar sel_2011
#     filename = re.sub(r"^sel_\d+", "", filename)

#     # eliminar guiones bajo al principio y final
#     filename = filename[:-4].strip("_") + ".pdf"

#     # renombrar extra, ord a junio, julio, septiembre
#     year = int(file.parts[-2][:4])

#     if year <= 2020:
#             filename = (filename
#                             .replace("extra_", "septiembre_")
#                             .replace("extraordinaria_", "septiembre_")
#                             .replace("ordinaria_", "junio_")
#                             .replace("ord_", "junio_")
                        
#             )
#     else:
#         filename = (filename
#                         .replace("extra_", "julio_")
#                         .replace("extraordinaria_", "julio_")
#                         .replace("ordinaria_", "junio_")
#                         .replace("ord_", "junio_")     
#         )

#     # eliminar "ua" (universidad almería)
#     filename = filename.replace("ua", "")
    
#     # eliminar palabras duplicadas
#     filename_words = filename.split("_")
#     unique_filename_words = []
#     for w in filename_words:
#         if w not in unique_filename_words:
#             unique_filename_words.append(w)
    
#     filename = "_".join(unique_filename_words)

#     # añadir año y asignatura al nombre de archivo
#     filename = f"{str(year)}_quimica_{filename}"

#     Path.mkdir(Path(f"../examenes_quimica_renombrados/{file.parts[-2]}"), parents=True, exist_ok=True)
#     new_file = f"../examenes_quimica_renombrados/{file.parts[-2]}/{filename}"
#     shutil.copy(file, new_file)

In [17]:
# old_filenames = ["/".join(file.parts[-2:]) for file in get_files()]
# new_filenames = ["/".join(file.parts[-2:]) for file in Path("../examenes_quimica_renombrados").rglob("*.pdf")]
# filenames = [(old, new) for old, new in zip(old_filenames, new_filenames)]
# filenames_df = pd.DataFrame(filenames, columns = ["old_filename", "new_filename"])

In [18]:
# filenames_df

In [19]:
# filenames_df.to_csv(ROOT / "filenames.csv", index = False)

In [20]:
# for folder in sorted((ROOT / "examenes_quimica").iterdir()):
#     if folder.is_dir():
#         files_count = len(list(folder.iterdir()))
#         if  files_count != 6:
#             print(f"Error en carpeta {folder}. {files_count} archivos.")

- 2012 tiene las opciones por separado (opciones A y B). Además, en dos de los exámenes están las opciones por separado.y un pdf con las dos opciones juntas. Requiere manipulación de los ejercicios.
  
- 2016 tiene solo 5 ficheros. Falta un examen.